# PowerWise: from neuron to forecast
This notebook walks from one neuron to a time-aware electricity demand experiment. The default series is synthetic, not utility data.

In [ ]:
import numpy as np
from src.activations import relu, sigmoid
from src.data import make_synthetic_demand, make_features, chronological_split, Standardizer
from src.network import NumpyMLP, mse

# One neuron: z = x·w + b, then activation
x = np.array([2., 1.]); w = np.array([1.5, -1.]); b = 0.5
z = x @ w + b
print('z:', z, 'ReLU(z):', relu(z))

## Feedforward shapes
For a batch with six features and eight hidden neurons: `X (batch,6) @ W1 (6,8) + b1 (1,8) -> A1 (batch,8) @ W2 (8,1) + b2 (1,1)`. The main network computes all rows with NumPy matrix multiplication.

In [ ]:
frame = make_synthetic_demand(hours=24*30)
X, y = make_features(frame)
xt, yt, xv, yv, xe, ye = chronological_split(X, y)
scaler = Standardizer().fit(xt.to_numpy())  # fit training window only
train_x, val_x, test_x = [scaler.transform(part.to_numpy()) for part in (xt, xv, xe)]
model = NumpyMLP(input_size=train_x.shape[1], hidden_size=8, seed=7)
history = model.fit(train_x, yt.to_numpy(), epochs=100, learning_rate=.005,
                    batch_size=32, seed=7, validation=(val_x, yv.to_numpy()))
print('train MSE:', history.train_loss[-1], 'validation MSE:', history.validation_loss[-1])
print('test MSE:', mse(ye.to_numpy(), model.forward(test_x)[0]))

## Interpret carefully
This chronological holdout is later than training data. Its synthetic-data scores are only a software demonstration. Replace the generated series with a documented dataset and check that every feature is available at prediction time before making real-world claims.